# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [57]:
# Write your code below.
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [58]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [59]:
# Write your code below.
import os
from glob import glob

price_data_files = glob(os.path.join(os.getenv("PRICE_DATA"),"**/*.parquet"), recursive=True)
print(f'Loaded {len(price_data_files)} parquet files from price data')

price_data = dd.read_parquet(price_data_files).set_index('ticker')
price_data

Loaded 3417 parquet files from price data


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year
npartitions=90,,,,,,,,,
AGE,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32
AIZ,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...
ZAGG,...,...,...,...,...,...,...,...,...
ZAGG,...,...,...,...,...,...,...,...,...


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [60]:
# Write your code below.
def multiple_transforms(x):
    return x.assign(
        Close_lag_1 = x['Close'].shift(1),
        Close_lag_adj = x['Adj Close'].shift(1),
    )

dd_feat = price_data.groupby('ticker', group_keys=False).apply(multiple_transforms)
dd_feat = dd_feat.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1,
    hi_lo_range = lambda x: x['High'] - x['Low']
)
dd_feat

/tmp/ipykernel_24040/3657837365.py:8: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = price_data.groupby('ticker', group_keys=False).apply(multiple_transforms)


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Close_lag_adj,Returns,hi_lo_range
npartitions=90,,,,,,,,,,,,,
AGE,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32,float64,float64,float64,float64
AIZ,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZAGG,...,...,...,...,...,...,...,...,...,...,...,...,...
ZAGG,...,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [61]:
# Write your code below.

# Dask optimized version
#result = dd_feat.groupby('ticker', group_keys=False).apply(
#    lambda x: x.assign(moving_average = x['Returns'].rolling(10).mean())
#).compute()

dd_feat_computed = dd_feat.compute()
result = dd_feat_computed.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(moving_average = x['Returns'].rolling(10).mean())
)
result

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Close_lag_adj,Returns,hi_lo_range,moving_average
ticker,,,,,,,,,,,,,,
AGE,2000-01-03,19.235901,20.472500,19.235901,20.289301,20.289301,2402.0,AGE.csv,2000,NaN,NaN,NaN,1.236599,NaN
AGE,2000-01-04,19.831301,20.335100,19.831301,19.877100,19.877100,7860.0,AGE.csv,2000,20.289301,20.289301,-0.020316,0.503799,NaN
AGE,2000-01-05,19.510700,19.693899,19.235901,19.327499,19.327499,3493.0,AGE.csv,2000,19.877100,19.877100,-0.027650,0.457998,NaN
AGE,2000-01-06,19.327499,19.006901,19.006901,19.006901,19.006901,109.0,AGE.csv,2000,19.327499,19.327499,-0.016588,0.000000,NaN
AGE,2000-01-07,18.548901,19.602301,18.548901,19.327499,19.327499,873.0,AGE.csv,2000,19.006901,19.006901,0.016867,1.053400,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZAGG,2020-03-26,3.100000,3.440000,2.910000,3.430000,3.430000,779100.0,ZAGG.csv,2020,3.080000,3.080000,0.113636,0.530000,0.045247
ZAGG,2020-03-27,3.300000,3.350000,3.010000,3.180000,3.180000,407500.0,ZAGG.csv,2020,3.430000,3.430000,-0.072886,0.340000,0.017794
ZAGG,2020-03-30,3.260000,3.450000,3.040000,3.450000,3.450000,1022200.0,ZAGG.csv,2020,3.180000,3.180000,0.084906,0.410000,0.042038


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
No, is not necessary. We can do the same calculations with Dask before computing and take advantage of parallel processing.

+ Would it have been better to do it in Dask? Why?
Yes is better because it will execute faster and it will lazy evaluate the operations only when results are needed.

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.